
# GVH Diagonal Cubic 0.3.2.7.3.7.2.1 — Exact GVH Tensor-Coefficient Lift and Full-Field Kinetic Hessian

**Auteur :** Charlemagne O Laurince  
**Version :** 0.3.2.7.3.7.2.1

## Mission

`0.3.2.7.3.7.2` a démontré l'inversion d'un modèle cinétique couplé paramétré,
mais ses coefficients \(\alpha,\beta,g_1,g_2,g_3\) n'étaient pas encore les coefficients tensoriels exacts de GVH.

Ici on repart des blocs ADM directionnels déjà établis :

\[
A=-\mathcal D_\perp s-v^ia_i^{(n)},
\]

\[
B_i=s\,a_i^{(n)}+\mathcal D_\perp v_i-K_i{}^jv_j,
\]

\[
C_i=-D_is-K_i{}^jv_j,
\]

\[
D_{ij}=D_iv_j+sK_{ij},
\]

et de

\[
\mathcal L_u=-c_1I_1-c_2\theta^2-c_3I_3+c_4a^2.
\]

Le but est de calculer le Hessien cinétique directement à partir de ces expressions,
sans réintroduire de coefficients proxy.

### Portée

Pour rendre le calcul symbolique contrôlable, on travaille **au point local ADM**
dans un repère spatial orthonormal :

\[
h_{ij}=\delta_{ij},\qquad N=1,
\]

en conservant :

- les 6 composantes indépendantes de \(K_{ij}\),
- \(\mathcal D_\perp s\),
- les 3 composantes de \(\mathcal D_\perp v_i\),
- \(s\) et \(v_i\) arbitraires,
- \(c_1,c_2,c_3,c_4\) arbitraires.

Les gradients spatiaux et l'accélération du normal peuvent contribuer aux termes linéaires/potentiels, mais **pas au Hessien par rapport aux vitesses**. Ils sont donc mis à zéro uniquement pour l'extraction du bloc cinétique.

\[
\boxed{\mathrm{DISPERSION\_READY=False}}
\]


In [13]:

import sympy as sp, json
from pathlib import Path
print("GVH 0.3.2.7.3.7.2.1")
print("SymPy:", sp.__version__)


GVH 0.3.2.7.3.7.2.1
SymPy: 1.14.0



## 1. Variables exactes de la branche cinétique locale

On utilise

\[
V^A=
(K_{11},K_{22},K_{33},K_{12},K_{13},K_{23},S,W_1,W_2,W_3),
\]

avec

\[
S=\mathcal D_\perp s,
\qquad
W_i=\mathcal D_\perp v_i.
\]

Le vecteur spatial est

\[
v_i=(v_1,v_2,v_3).
\]


In [14]:

c1,c2,c3,c4 = sp.symbols("c1 c2 c3 c4", real=True)
s = sp.symbols("s", real=True)
v1,v2,v3 = sp.symbols("v1 v2 v3", real=True)

K11,K22,K33,K12,K13,K23,S,W1,W2,W3 = sp.symbols(
    "K11 K22 K33 K12 K13 K23 S W1 W2 W3", real=True
)

K = sp.Matrix([
    [K11,K12,K13],
    [K12,K22,K23],
    [K13,K23,K33]
])
v = sp.Matrix([v1,v2,v3])
W = sp.Matrix([W1,W2,W3])

vel = sp.Matrix([K11,K22,K33,K12,K13,K23,S,W1,W2,W3])
print("velocity dimension =", len(vel))
assert len(vel)==10


velocity dimension = 10



## 2. Lift exact des blocs \(A,B_i,C_i,D_{ij}\)

Pour l'extraction du Hessien cinétique au point local :

\[
a_i^{(n)}=0,\qquad D_is=0,\qquad D_iv_j=0.
\]

Ainsi, la **partie dépendante des vitesses** est exactement :

\[
A_{\rm kin}=-S,
\]

\[
B_i^{\rm kin}=W_i-K_i{}^jv_j,
\]

\[
C_i^{\rm kin}=-K_i{}^jv_j,
\]

\[
D_{ij}^{\rm kin}=sK_{ij}.
\]

Aucun coefficient proxy n'est introduit.


In [15]:

A = -S
B = W - K*v
Cc = -K*v
D = s*K

print("A =", A)
print("B =", B.T)
print("C =", Cc.T)


A = -S
B = Matrix([[-K11*v1 - K12*v2 - K13*v3 + W1, -K12*v1 - K22*v2 - K23*v3 + W2, -K13*v1 - K23*v2 - K33*v3 + W3]])
C = Matrix([[-K11*v1 - K12*v2 - K13*v3, -K12*v1 - K22*v2 - K23*v3, -K13*v1 - K23*v2 - K33*v3]])



## 3. Invariants cinétiques ADM

Dans le split orthonormal local, on reconstruit les contractions cinétiques sous la forme :

\[
I_1^{\rm kin}
=
-A^2+B_iB_i-C_iC_i+D_{ij}D_{ij},
\]

\[
\theta_{\rm kin}
=
-A+\operatorname{tr}D,
\]

\[
I_3^{\rm kin}
=
-A^2+2B_iC_i+D_{ij}D_{ji},
\]

et, pour la partie dépendante des vitesses de l'accélération du vecteur,

\[
a_i^{(u),\rm kin}=sB_i+v^jD_{ji},
\]

\[
a_0^{(u),\rm kin}=v^iB_i.
\]

On prend alors

\[
a_{\rm kin}^2
=
-(a_0^{(u),\rm kin})^2
+
a_i^{(u),\rm kin}a_i^{(u),\rm kin}.
\]

Ces expressions sont utilisées ici comme **lift tensoriel ADM candidat exact du secteur cinétique**, issu des blocs de 0.3.2.7.3.4. Le notebook audite ce lift et ne le confond pas avec une dérivation 4D indépendante supplémentaire.


In [16]:

def frob2(M):
    return sp.expand(sum(M[i,j]**2 for i in range(M.rows) for j in range(M.cols)))

I1 = sp.expand(-A**2 + (B.dot(B)) - (Cc.dot(Cc)) + frob2(D))
theta = sp.expand(-A + sp.trace(D))
I3 = sp.expand(-A**2 + 2*(B.dot(Cc)) + sp.trace(D*D))

a_sp = sp.expand(s)*B + D.T*v
a0 = sp.expand(v.dot(B))
a2 = sp.expand(-a0**2 + a_sp.dot(a_sp))

Lu_kin = sp.expand(-c1*I1 - c2*theta**2 - c3*I3 + c4*a2)

print("Quadratic degree check:")
poly = sp.Poly(Lu_kin, *list(vel))
print("total degree =", poly.total_degree())
assert poly.total_degree() <= 2


Quadratic degree check:
total degree = 2



## 4. Hessien cinétique GVH local exact

On définit :

\[
\boxed{
Q^{\rm GVH}_{AB}
=
\frac{\partial^2\mathcal L_{u,\rm kin}}
{\partial V^A\,\partial V^B}.
}
\]

Le Hessien est calculé directement depuis \(\mathcal L_{u,\rm kin}\).


In [17]:

Q_u = sp.simplify(sp.hessian(Lu_kin, list(vel)))

assert Q_u.shape == (10,10)
assert sp.simplify(Q_u-Q_u.T)==sp.zeros(10)

print("Q_u shape =", Q_u.shape)
print("symmetric =", sp.simplify(Q_u-Q_u.T)==sp.zeros(10))


Q_u shape = (10, 10)
symmetric = True



## 5. Contrôle \(v_i=0\)

La branche alignée locale

\[
v_i=0
\]

fournit un contrôle analytique important. Le Hessien doit alors se simplifier fortement et exposer les combinaisons des \(c_i\) sans mélange directionnel dû à \(v_i\neq0\).


In [18]:

subs_v0 = {v1:0,v2:0,v3:0}
Q_v0 = sp.simplify(Q_u.subs(subs_v0))

rank_v0_generic = Q_v0.subs({
    c1:2,c2:sp.Rational(1,3),c3:sp.Rational(2,5),c4:sp.Rational(1,7),
    s:1
}).rank()

print("generic witness rank at v=0 =", rank_v0_generic)


generic witness rank at v=0 = 10



## 6. Branche générique non alignée — témoin exact

Pour éviter d'affirmer symboliquement un déterminant gigantesque non factorisé,
on utilise deux niveaux :

1. le Hessien \(Q_u\) est **symbolique exact** dans \((s,v_i,c_i)\);
2. une substitution rationnelle générique vérifie si le rang maximal \(10\) est effectivement accessible.

Un témoin de rang maximal prouve l'existence d'une branche ouverte non dégénérée,
mais ne donne pas encore la factorisation complète de toutes les surfaces de dégénérescence.


In [19]:

generic_subs = {
    c1:sp.Rational(7,5),
    c2:sp.Rational(2,7),
    c3:sp.Rational(3,11),
    c4:sp.Rational(5,13),
    s:sp.Rational(6,5),
    v1:sp.Rational(1,5),
    v2:sp.Rational(1,7),
    v3:sp.Rational(1,11),
}
Q_num = sp.Matrix(Q_u.subs(generic_subs))
rank_generic = Q_num.rank()
det_generic = sp.factor(Q_num.det())

print("generic exact-rational rank =", rank_generic)
print("generic exact-rational determinant nonzero =", det_generic != 0)
assert rank_generic == 10
assert det_generic != 0


generic exact-rational rank = 10
generic exact-rational determinant nonzero = True



## 7. Extraction de sous-blocs physiques

On décompose

\[
Q_u=
\begin{pmatrix}
Q_{KK} & Q_{KX}\\
Q_{XK} & Q_{XX}
\end{pmatrix},
\]

où \(K\) contient les 6 vitesses métriques et

\[
X=(S,W_1,W_2,W_3).
\]

Cette décomposition permet d'identifier si les modes directionnels sont directement couplés aux vitesses métriques.


In [20]:

Q_KK = Q_u[:6,:6]
Q_KX = Q_u[:6,6:]
Q_XX = Q_u[6:,6:]

print("Q_KK:",Q_KK.shape)
print("Q_KX:",Q_KX.shape)
print("Q_XX:",Q_XX.shape)

coupling_zero_v0 = sp.simplify(Q_KX.subs(subs_v0))
print("K-X coupling at v=0 identically zero? ", coupling_zero_v0==sp.zeros(6,4))


Q_KK: (6, 6)
Q_KX: (6, 4)
Q_XX: (4, 4)
K-X coupling at v=0 identically zero?  False



## 8. Complément de Schur sur témoin générique

Si \(Q_{KK}\) est inversible :

\[
S_X=Q_{XX}-Q_{XK}Q_{KK}^{-1}Q_{KX}.
\]

On vérifie cette structure sur un témoin rationnel générique.


In [21]:

QKK_num = Q_KK.subs(generic_subs)
QKX_num = Q_KX.subs(generic_subs)
QXX_num = Q_XX.subs(generic_subs)

rank_QKK = QKK_num.rank()
print("rank Q_KK witness =", rank_QKK)

if rank_QKK == 6:
    Schur_num = sp.simplify(QXX_num - QKX_num.T*QKK_num.inv()*QKX_num)
    print("rank Schur witness =", Schur_num.rank())
    assert Schur_num.rank()==4
    assert sp.simplify(QKK_num.det()*Schur_num.det()-Q_num.det())==0
    SCHUR_PASS=True
else:
    SCHUR_PASS=False

print("Schur factorization witness =", SCHUR_PASS)
assert SCHUR_PASS


rank Q_KK witness = 6
rank Schur witness = 4
Schur factorization witness = True



## 9. Inversion générique exacte au témoin

Le témoin rationnel de rang \(10\) permet une inversion exacte :

\[
V^A=(Q_u^{-1})^{AB}P_B.
\]

Cette cellule vérifie l'identité sans approximation flottante.


In [22]:

Qinv_num = Q_num.inv()
assert sp.simplify(Q_num*Qinv_num-sp.eye(10))==sp.zeros(10)

P0 = sp.Matrix(sp.symbols("P0:10"))
Vsol_num = Qinv_num*P0
print("exact rational witness inverse: PASS")
print("number of solved velocities =",len(Vsol_num))


exact rational witness inverse: PASS
number of solved velocities = 10



## 10. Ce que le notebook démontre et ce qu'il ne démontre pas

### Démontré ici

- le lift cinétique est écrit directement en \(c_1,c_2,c_3,c_4,s,v_i\), sans \(\alpha,\beta,g_i\);
- le Hessien \(10\times10\) est dérivé symboliquement;
- il est symétrique;
- une branche rationnelle exacte de rang \(10\) existe;
- le complément de Schur et l'inversion fonctionnent sur cette branche.

### Encore ouvert

- factorisation symbolique complète de \(\det Q_u\);
- classification de toutes les surfaces de dégénérescence;
- comparaison indépendante avec une variation 4D full-field non réduite;
- ajout du Hessien Einstein-Hilbert total au lieu du seul secteur \(u\);
- densités canoniques finales \(\mathcal C_\perp,\mathcal C_i\);
- algèbre hypersurface.


In [23]:

GATES = {
    "proxy_coefficients_removed":True,
    "ADM_block_lift_explicit":True,
    "symbolic_10x10_directional_Hessian":True,
    "Hessian_symmetric":True,
    "generic_exact_rational_rank10_witness":True,
    "generic_exact_rational_inverse":True,
    "Schur_witness_pass":True,

    "full_symbolic_det_factorization":False,
    "all_degeneracy_surfaces_classified":False,
    "independent_unreduced_4D_crosscheck":False,
    "EH_plus_directional_total_Hessian":False,
    "explicit_full_Cperp":False,
    "explicit_full_Ci":False,
    "hypersurface_algebra_closed":False,
}

for k,v in GATES.items():
    print(k,":",v)

CORE_PASS = all(list(GATES.values())[:7])
FULL_PASS = all(GATES.values())
assert CORE_PASS
assert not FULL_PASS

FINAL_STATUS = (
    "PARTIAL-PASS-EXACT-CI-TENSOR-COEFFICIENT-LIFT-AND-SYMBOLIC-10X10-DIRECTIONAL-HESSIAN_"
    "GENERIC-RANK10-BRANCH-EXISTS_"
    "BLOCKED-TOTAL-EH-PLUS-U-HESSIAN-DEGENERACY-CLASSIFICATION-AND-HYPERSURFACE-ALGEBRA"
)
DISPERSION_READY=False

print("\nFINAL STATUS:",FINAL_STATUS)
print("DISPERSION_READY =",DISPERSION_READY)


proxy_coefficients_removed : True
ADM_block_lift_explicit : True
symbolic_10x10_directional_Hessian : True
Hessian_symmetric : True
generic_exact_rational_rank10_witness : True
generic_exact_rational_inverse : True
Schur_witness_pass : True
full_symbolic_det_factorization : False
all_degeneracy_surfaces_classified : False
independent_unreduced_4D_crosscheck : False
EH_plus_directional_total_Hessian : False
explicit_full_Cperp : False
explicit_full_Ci : False
hypersurface_algebra_closed : False

FINAL STATUS: PARTIAL-PASS-EXACT-CI-TENSOR-COEFFICIENT-LIFT-AND-SYMBOLIC-10X10-DIRECTIONAL-HESSIAN_GENERIC-RANK10-BRANCH-EXISTS_BLOCKED-TOTAL-EH-PLUS-U-HESSIAN-DEGENERACY-CLASSIFICATION-AND-HYPERSURFACE-ALGEBRA
DISPERSION_READY = False



## 11. Prochaine étape

Le résultat indique qu'une branche non dégénérée existe pour le Hessien directionnel exact local.

La prochaine sous-étape doit être :

### `0.3.2.7.3.7.2.2 — Total EH+Directional Kinetic Hessian, Determinant Factorization and Degeneracy Surfaces`

Elle devra :

1. ajouter le bloc cinétique Einstein-Hilbert au Hessien directionnel;
2. obtenir
   \[
   Q_{\rm total}=Q_{\rm EH}+Q_u;
   \]
3. factoriser autant que possible
   \[
   \det Q_{\rm total};
   \]
4. identifier les combinaisons de \(c_i\) responsables des pertes de rang;
5. séparer la branche générique des branches dégénérées;
6. préparer l'inversion totale pour construire ensuite
   \[
   \mathcal C_\perp,\mathcal C_i.
   \]

Toujours :

\[
\boxed{\mathrm{DISPERSION\_READY=False}}.
\]


In [24]:

artifact = {
    "notebook":"GVH_Diagonal_Cubic_0.3.2.7.3.7.2.1",
    "final_status":FINAL_STATUS,
    "velocity_dimension":10,
    "generic_exact_rational_rank":int(rank_generic),
    "generic_exact_rational_det_nonzero":bool(det_generic != 0),
    "rank_QKK_witness":int(rank_QKK),
    "schur_witness_pass":bool(SCHUR_PASS),
    "proxy_coefficients_removed":True,
    "gates":GATES,
    "dispersion_ready":False,
    "next":"GVH_Diagonal_Cubic_0.3.2.7.3.7.2.2_Total_EH_Directional_Kinetic_Hessian_Determinant_Factorization_and_Degeneracy_Surfaces.ipynb"
}

export_dir=Path("/content/gvh_exports") if Path("/content").exists() else Path.cwd()/"gvh_exports"
export_dir.mkdir(parents=True,exist_ok=True)
artifact_path=export_dir/"gvh_0.3.2.7.3.7.2.1_exact_tensor_lift_hessian.json"
artifact_path.write_text(json.dumps(artifact,indent=2),encoding="utf-8")
print("Artifact:",artifact_path)


Artifact: /content/gvh_exports/gvh_0.3.2.7.3.7.2.1_exact_tensor_lift_hessian.json



# Conclusion

`0.3.2.7.3.7.2.1` retire les coefficients proxy du notebook précédent et construit le Hessien cinétique directionnel directement à partir des blocs ADM GVH.

Un témoin rationnel exact montre :

\[
\boxed{\operatorname{rank}Q_u=10}
\]

sur une branche générique non alignée.

Ainsi, le secteur cinétique directionnel candidat n'est pas identiquement dégénéré.

Mais le Hessien **total** incluant Einstein-Hilbert, sa factorisation complète, les surfaces de dégénérescence et l'algèbre des contraintes restent ouverts.

Verdict :

\[
\boxed{\text{PARTIAL PASS}}
\]

et

\[
\boxed{\mathrm{DISPERSION\_READY=False}}.
\]
